In [527]:
import pandas as pd
import numpy as np
import os
import re

from eval_plot_utils import parse_yml_pred,output_evaluation_outputs


In [528]:
pairs_aug = pd.read_csv('/ksi_ml/data/pairs_aug_ptmvar.csv').iloc[:,1:].drop_duplicates()
# tape_pred = parse_yml_pred('/ksi_ml/code/evaluation/eval_outputs/exp_tape_bertbase_sep_v2__ptmvar__1b97f9a11dd9.pkl',
#                           return_logits=True).drop_duplicates()

tape_pred = parse_yml_pred('/ksi_ml/code/evaluation/eval_outputs/exp_tape_bertbase_sep_v2_long__ptmvar_on_psite__6eda71d75e35.pkl',
                          return_logits=True).drop_duplicates()

In [529]:
tape_pred.columns = ['kinase_domain_seq','substrate_seq_15aa','probs','logits']
tape_pred = pd.merge(tape_pred,pairs_aug)
tape_pred = tape_pred.loc[:, ['kinase','kinase_gene','substrate','p_ind','p_aa','substrate_seq_15aa','probs','logits']]

In [530]:
tape_pred = tape_pred.drop_duplicates(['kinase','substrate_seq_15aa'])

In [531]:
kinome_dat = pd.read_pickle('/kinase_ml/data/human_kinome_seq_230301.pkl')

In [532]:
kinome_dat['p_aa'] = 'ST'
kinome_dat.loc[kinome_dat['pk_group']=='TK','p_aa'] = 'Y'

In [533]:
tape_pred['kinase_gene'].drop_duplicates().isin(kinome_dat['kinase_gene']).all()

np.True_

In [536]:
df = tape_pred

In [537]:
atypical_kinase = kinome_dat.loc[kinome_dat['pk_group']=='Atypical','kinase_gene'].tolist()

In [538]:
exclude_ls = ['CDC','CDK','ADCK','DYRK']
df = df.loc[df['kinase_gene'].apply(lambda s: np.all([elem not in s for elem in exclude_ls])),:]

df = df.loc[~df['kinase_gene'].isin(atypical_kinase),:]

In [539]:
import numpy as np
import pandas as pd

LOW_LOGIT_THRESHOLD = 5

df = df.copy()

# --- ranks within kinase_gene (competition ranking) ---
df["logit_rank_within_kinase"] = (
    df.groupby("kinase_gene")["logits"]
      .rank(method="min", ascending=False)
)
df["n_rows_within_kinase"] = df.groupby("kinase_gene")["logits"].transform("size")
df["high_rank_cutoff"] = np.ceil(0.05 * df["n_rows_within_kinase"]).astype(int)

# --- low & high flags ---
df["is_low_kinase"] = df["logits"] < LOW_LOGIT_THRESHOLD

# ✅ high must be top-20% AND not low (i.e., meets the logit floor)
df["is_high_kinase"] = (
    (df["logit_rank_within_kinase"] <= df["high_rank_cutoff"]) &
    (~df["is_low_kinase"])
)

# --- categorical label: low takes precedence over high ---
df["logit_category"] = np.select(
    [
        df["is_low_kinase"],
        df["is_high_kinase"],
    ],
    ["low", "high"],
    default=""
)

# --- kinase:logit:rank string ---
df["kinase_logit_rank"] = (
    df["kinase_gene"].astype(str) + ":" +
    df["logits"].round(3).astype(str) + ":" +
    df["logit_rank_within_kinase"].astype(int).astype(str)
)

def join_sorted(x: pd.Series) -> str:
    # sort by rank (last field): KINASE:LOGIT:RANK
    return "; ".join(
        x.sort_values(
            key=lambda s: s.str.extract(r":([-0-9]+)$")[0].astype(int),
            ascending=True
        )
    )

# ---- substrate_stats ----
substrate_stats = df.groupby("substrate_seq_15aa").agg(
    total_kinases=("logits", "count"),
    mean_logits=("logits", "mean"),
    std_logits=("logits", "std"),
    min_logits=("logits", "min"),
    max_logits=("logits", "max"),
    mean_probs=("probs", "mean"),
    max_probs=("probs", "max"),
).reset_index()

high_low_counts = df.groupby("substrate_seq_15aa").agg(
    high_rank_kinases=("is_high_kinase", "sum"),
    low_logit_kinases=("is_low_kinase", "sum"),
).reset_index()

substrate_stats = substrate_stats.merge(high_low_counts, on="substrate_seq_15aa", how="left")
substrate_stats["high_kinase_ratio"] = substrate_stats["high_rank_kinases"] / substrate_stats["total_kinases"]
substrate_stats["low_kinase_ratio"] = substrate_stats["low_logit_kinases"] / substrate_stats["total_kinases"]
substrate_stats["logit_range"] = substrate_stats["max_logits"] - substrate_stats["min_logits"]

# ---- lists ----
substrate_kinase_lists = (
    df.loc[df["logit_category"].isin(["high", "low"])]
      .groupby(["substrate_seq_15aa", "logit_category"])["kinase_logit_rank"]
      .apply(join_sorted)
      .unstack(fill_value="")
      .reset_index()
      .rename(columns={"high": "high_kinases", "low": "low_kinases"})
)

# overwrite-safe merge (prevents duplicate column issues on rerun)
substrate_stats = substrate_stats.drop(columns=["high_kinases", "low_kinases"], errors="ignore")
substrate_stats = substrate_stats.merge(substrate_kinase_lists, on="substrate_seq_15aa", how="left")
substrate_stats["high_kinases"] = substrate_stats["high_kinases"].fillna("")
substrate_stats["low_kinases"] = substrate_stats["low_kinases"].fillna("")


In [540]:
substrate_stats = substrate_stats.drop_duplicates(['substrate_seq_15aa'])

In [541]:
substrate_stats = pd.merge(pairs_aug,substrate_stats,on = 'substrate_seq_15aa')

In [542]:
substrate_stats = substrate_stats.loc[:,['substrate_seq_15aa','substrate_gene','p_aa','p_ind',
                       'var_type','var_class','dbSNP','FTID',
       'high_kinase_ratio', 'low_kinase_ratio', 'logit_range',
                                         'high_kinases','low_kinases',]]

In [543]:
substrate_stats.head(1)

,substrate_seq_15aa,substrate_gene,p_aa,p_ind,var_type,var_class,dbSNP,FTID,high_kinase_ratio,low_kinase_ratio,logit_range,high_kinases,low_kinases
0,QGCVYVQTVLRYNIL,A2ML1,T,1312,LB/B,I,rs201083574,VAR_071856,0.0,1.0,7.833984,,MYO3B:0.365:239; MYO3A:-2.172:422; HGK:-3.312:...


In [544]:
pairs_aug.loc[pairs_aug['substrate_gene'] == 'CTNNB1',['dbSNP','p_aa','p_ind']].drop_duplicates()

,dbSNP,p_aa,p_ind
320,rs1413975856,S,23
321,rs121913409,S,45
322,rs121913407,S,45
323,rs121913412,T,41
324,rs121913413,T,41
325,rs121913400,S,33
326,-,S,33
327,rs121913228,S,37
328,rs121913403,S,37


In [545]:
substrate_stats = substrate_stats.sort_values('dbSNP').groupby(['substrate_seq_15aa']).tail(1)

In [546]:
# recapture GSK3a/b on CTNNB1
substrate_stats.loc[substrate_stats['substrate_gene'] == 'CTNNB1',:]

,substrate_seq_15aa,substrate_gene,p_aa,p_ind,var_type,var_class,dbSNP,FTID,high_kinase_ratio,low_kinase_ratio,logit_range,high_kinases,low_kinases
715977,QQQSYLDSGIHSGAT,CTNNB1,S,33,US,I,rs121913400,VAR_017617,0.052863,0.947137,14.37500,GSK3A:5.375:2; GSK3B:5.062:3; HIPK2:6.75:3; HI...,Erk2:4.906:23; Erk3:4.438:79; PITSLRE:4.125:98...
444468,YLDSGIHSGATTTAP,CTNNB1,S,37,LP/P,I,rs121913403,VAR_017625,0.052863,0.947137,14.12500,HIPK1:6.844:3; JNK1:7.156:3; HIPK2:6.656:4; Er...,GSK3B:4.625:6; GSK3A:4.719:7; PRP4:4.938:60; P...
32165,GATTTAPSLSGKGNP,CTNNB1,S,45,US,I,rs121913409,VAR_017631,0.019824,0.973568,11.46875,CK1e:6.906:9; CK1a2:6.062:10; CK1d:6.875:11; C...,BMPR2:4.875:58; TBK1:4.906:60; NDR1:3.75:61; N...
17084,GIHSGATTTAPSLSG,CTNNB1,T,41,US,I,rs121913413,VAR_017630,0.052863,0.942731,12.71875,p38a:5.75:12; CHED:5.75:17; p38b:6.219:19; PRP...,CK1a:4.906:19; GSK3B:3.953:21; GSK3A:3.688:25;...
669044,PDRKAAVSHWQQQSY,CTNNB1,S,23,US,I,rs1413975856,VAR_017612,0.004405,0.995595,12.50000,PRP4:5.344:39; HIPK4:5.75:54,Erk5:3.969:90; IKKe:3.891:100; MSSK1:3.562:129...


In [547]:
# Filter for substrates with few high-scoring and many low-scoring kinases
# CRITICAL: Must have at least 1 high-scoring kinase (> 0)
selective_substrates = substrate_stats[
    (substrate_stats['high_kinases'] !='') &  # MUST HAVE at least 1 high scorer
    (substrate_stats['low_kinase_ratio'] >= 0.9) 
].sort_values('high_kinase_ratio', ascending=True)

In [548]:
import requests
import pandas as pd
from typing import List, Dict
import time

def query_rsid_diseases(rsids: List[str], api: str = 'myvariant') -> pd.DataFrame:
    """
    Query disease associations for a list of rsIDs.
    
    Parameters:
    -----------
    rsids : List[str]
        List of rsIDs to query (e.g., ["rs958068", "rs121913400"])
    api : str
        Which API to use: 'myvariant' (default) or 'ensembl'
    
    Returns:
    --------
    pd.DataFrame
        DataFrame with columns: rsID, Clinical_Significance, Disease/Phenotype, 
        Review_Status, Accession
    """
    
    if api == 'myvariant':
        return _query_myvariant(rsids)
    elif api == 'ensembl':
        return _query_ensembl(rsids)
    else:
        raise ValueError("api must be 'myvariant' or 'ensembl'")


def _query_myvariant(rsids: List[str]) -> pd.DataFrame:
    """Query using MyVariant.info API"""
    base_url = "https://myvariant.info/v1/query"
    results = []
    
    for rsid in rsids:
        # Ensure rsid has 'rs' prefix
        if not rsid.startswith('rs'):
            rsid = f'rs{rsid}'
        
        try:
            # Query MyVariant.info
            response = requests.get(
                base_url,
                params={
                    'q': f'dbsnp.rsid:{rsid}',
                    'fields': 'dbsnp.rsid,clinvar,dbsnp.gene'
                },
                timeout=10
            )
            
            if response.status_code == 200:
                data = response.json()
                
                # Handle single hit or multiple hits
                hits = data.get('hits', [])
                if not hits:
                    results.append({
                        'rsID': rsid,
                        'Clinical_Significance': 'No data found',
                        'Disease/Phenotype': 'N/A',
                        'Review_Status': 'N/A',
                        'Accession': 'N/A'
                    })
                    continue
                
                # Process each hit (variant)
                found_clinvar = False
                for hit in hits:
                    clinvar = hit.get('clinvar', {})
                    
                    if not clinvar:
                        continue
                    
                    found_clinvar = True
                    
                    # Handle both single entry and list of entries
                    if isinstance(clinvar, dict):
                        clinvar_entries = [clinvar]
                    else:
                        clinvar_entries = clinvar if isinstance(clinvar, list) else []
                    
                    for entry in clinvar_entries:
                        # Extract RCV information
                        rcv = entry.get('rcv', {})
                        
                        # RCV can also be a list
                        if isinstance(rcv, list):
                            rcv_list = rcv
                        else:
                            rcv_list = [rcv] if rcv else []
                        
                        for rcv_entry in rcv_list:
                            if not isinstance(rcv_entry, dict):
                                continue
                            
                            # Extract clinical significance
                            significance = rcv_entry.get('clinical_significance', 'Unknown')
                            
                            # Extract conditions/diseases
                            conditions = rcv_entry.get('conditions', {})
                            if isinstance(conditions, dict):
                                phenotype = conditions.get('name', 'Not specified')
                            elif isinstance(conditions, list):
                                phenotype = '; '.join([c.get('name', '') for c in conditions if isinstance(c, dict) and c.get('name')])
                                if not phenotype:
                                    phenotype = 'Not specified'
                            else:
                                phenotype = 'Not specified'
                            
                            # Get review status
                            review_status = rcv_entry.get('review_status', 'Not available')
                            
                            # Get accession
                            accession = rcv_entry.get('accession', 'N/A')
                            
                            results.append({
                                'rsID': rsid,
                                'Clinical_Significance': significance,
                                'Disease/Phenotype': phenotype,
                                'Review_Status': review_status,
                                'Accession': accession
                            })
                
                if not found_clinvar:
                    # No ClinVar data but variant exists
                    gene_info = hits[0].get('dbsnp', {}).get('gene', {})
                    gene_name = gene_info.get('symbol', 'Unknown') if isinstance(gene_info, dict) else 'Unknown'
                    
                    results.append({
                        'rsID': rsid,
                        'Clinical_Significance': 'No ClinVar data',
                        'Disease/Phenotype': f'Gene: {gene_name}',
                        'Review_Status': 'N/A',
                        'Accession': 'N/A'
                    })
            else:
                results.append({
                    'rsID': rsid,
                    'Clinical_Significance': f'Query failed (status {response.status_code})',
                    'Disease/Phenotype': 'N/A',
                    'Review_Status': 'N/A',
                    'Accession': 'N/A'
                })
        
        except Exception as e:
            results.append({
                'rsID': rsid,
                'Clinical_Significance': f'Error: {str(e)[:100]}',
                'Disease/Phenotype': 'N/A',
                'Review_Status': 'N/A',
                'Accession': 'N/A'
            })
        
        # Be respectful to the API
        time.sleep(0.3)
    
    # Create DataFrame
    df = pd.DataFrame(results)
    return df


def _query_ensembl(rsids: List[str]) -> pd.DataFrame:
    """Query using Ensembl REST API (alternative method)"""
    base_url = "https://rest.ensembl.org/variation/human"
    results = []
    
    for rsid in rsids:
        # Ensure rsid has 'rs' prefix
        if not rsid.startswith('rs'):
            rsid = f'rs{rsid}'
        
        try:
            response = requests.get(
                f"{base_url}/{rsid}",
                headers={"Content-Type": "application/json"},
                params={"phenotypes": 1},
                timeout=10
            )
            
            if response.status_code == 200:
                data = response.json()
                
                # Check for phenotype associations
                phenotypes = data.get('phenotypes', [])
                
                if phenotypes:
                    for pheno in phenotypes:
                        # Extract relevant information
                        trait = pheno.get('trait', 'Unknown')
                        source = pheno.get('source', 'Unknown')
                        clinical_significance = pheno.get('clinical_significance', 'Unknown')
                        
                        results.append({
                            'rsID': rsid,
                            'Clinical_Significance': clinical_significance if clinical_significance else 'Not specified',
                            'Disease/Phenotype': trait,
                            'Review_Status': 'N/A',
                            'Accession': f"Source: {source}"
                        })
                else:
                    results.append({
                        'rsID': rsid,
                        'Clinical_Significance': 'No phenotype data',
                        'Disease/Phenotype': 'N/A',
                        'Review_Status': 'N/A',
                        'Accession': 'N/A'
                    })
            else:
                results.append({
                    'rsID': rsid,
                    'Clinical_Significance': f'Query failed (status {response.status_code})',
                    'Disease/Phenotype': 'N/A',
                    'Review_Status': 'N/A',
                    'Accession': 'N/A'
                })
        
        except Exception as e:
            results.append({
                'rsID': rsid,
                'Clinical_Significance': f'Error: {str(e)[:100]}',
                'Disease/Phenotype': 'N/A',
                'Review_Status': 'N/A',
                'Accession': 'N/A'
            })
        
        # Be respectful to the API
        time.sleep(0.3)
    
    df = pd.DataFrame(results)
    return df


def group_by_rsid(df: pd.DataFrame) -> pd.DataFrame:
    """
    Group DataFrame by rsID, concatenating all associated information.
    
    Parameters:
    -----------
    df : pd.DataFrame
        DataFrame from query_rsid_diseases function
    
    Returns:
    --------
    pd.DataFrame
        Grouped DataFrame with one row per rsID
    """
    if df.empty:
        return df
    
    def concat_unique(series):
        """Concatenate unique non-N/A values"""
        unique_vals = series[~series.isin(['N/A', 'Not available', 'Unknown', 'Not specified'])].unique()
        if len(unique_vals) == 0:
            return 'N/A'
        return ' | '.join(unique_vals)
    
    grouped = df.groupby('rsID').agg({
        'Clinical_Significance': concat_unique,
        'Disease/Phenotype': concat_unique,
        'Review_Status': concat_unique,
        'Accession': concat_unique
    }).reset_index()
    
    return grouped
    

In [549]:
rsids = substrate_stats.loc[substrate_stats['dbSNP']!='-','dbSNP'].drop_duplicates().tolist()

In [550]:
print("Querying with MyVariant.info API...")
df = query_rsid_diseases(rsids, api='myvariant')

Querying with MyVariant.info API...


In [551]:
df_grouped = group_by_rsid(df)

In [552]:
len(rsids)

1169

In [553]:
selective_substrates = pd.merge(selective_substrates,df_grouped,left_on = 'dbSNP',right_on = 'rsID',how = 'left')

In [554]:
import requests
import pandas as pd
from typing import List, Tuple, Optional
import time

def query_omnipath_kinase_substrate(
    gene_pairs: List[Tuple[str, str]],
    effect_filter: Optional[str] = None
) -> pd.DataFrame:
    """
    Query OmniPath for kinase-substrate interactions matching the exact web interface format.
    
    Parameters:
    -----------
    gene_pairs : List[Tuple[str, str]]
        List of (source_gene, target_gene) tuples
        Example: [("GSK3B", "CTNNB1")]
    effect_filter : Optional[str]
        Filter by effect type: 'inhibition', 'stimulation', 'both', or None for all
    
    Returns:
    --------
    pd.DataFrame
        DataFrame matching OmniPath web interface format with columns:
        id, source, target, sourceGenesymbol, targetGenesymbol, isDirected, 
        isStimulation, isInhibition, consensusDirection, consensusStimulation,
        consensusInhibition, sources
    """
    
    base_url = "https://omnipathdb.org/interactions"
    
    all_results = []
    
    for source_gene, target_gene in gene_pairs:
        try:
            params = {
                'sources': source_gene.upper(),
                'targets': target_gene.upper(),
                'organisms': '9606',
                'genesymbols': '1',
                'datasets': 'omnipath',
                'license': 'academic'
            }
            
            response = requests.get(
                base_url,
                params=params,
                timeout=15
            )
            
            if response.status_code == 200:
                lines = response.text.strip().split('\n')
                
                if len(lines) <= 1:
                    continue
                
                # Parse header - first line contains column names
                header = lines[0].split('\t')
                
                # Parse each interaction
                for line in lines[1:]:
                    if not line.strip():
                        continue
                    
                    values = line.split('\t')
                    # Pad values if needed
                    values.extend([''] * (len(header) - len(values)))
                    
                    row_dict = dict(zip(header, values))
                    
                    # Build result with all available fields
                    result = {}
                    
                    # Map common field names
                    field_mapping = {
                        'source_genesymbol': 'sourceGenesymbol',
                        'target_genesymbol': 'targetGenesymbol',
                        'is_directed': 'isDirected',
                        'is_stimulation': 'isStimulation',
                        'is_inhibition': 'isInhibition',
                        'consensus_direction': 'consensusDirection',
                        'consensus_stimulation': 'consensusStimulation',
                        'consensus_inhibition': 'consensusInhibition',
                        'type':'type'
                    }
                    
                    # Copy all fields, applying mapping where available
                    for key, value in row_dict.items():
                        display_key = field_mapping.get(key, key)
                        result[display_key] = value
                    
                    # Convert 1/0 to TRUE/FALSE for boolean-looking columns
                    bool_columns = ['isDirected', 'isStimulation', 'isInhibition', 
                                  'consensusDirection', 'consensusStimulation', 'consensusInhibition',
                                  'is_directed', 'is_stimulation', 'is_inhibition',
                                  'consensus_direction', 'consensus_stimulation', 'consensus_inhibition']
                    
                    for col in bool_columns:
                        if col in result:
                            result[col] = 'TRUE' if result[col] == '1' else 'FALSE'
                    
                    # Apply effect filter if specified
                    if effect_filter:
                        consensus_inhib = result.get('consensusInhibition') or result.get('consensus_inhibition', 'FALSE')
                        consensus_stimul = result.get('consensusStimulation') or result.get('consensus_stimulation', 'FALSE')
                        
                        if effect_filter.lower() == 'inhibition' and consensus_inhib != 'TRUE':
                            continue
                        elif effect_filter.lower() == 'stimulation' and consensus_stimul != 'TRUE':
                            continue
                        elif effect_filter.lower() == 'both' and (consensus_inhib != 'TRUE' or consensus_stimul != 'TRUE'):
                            continue
                    
                    all_results.append(result)
            
            else:
                print(f"Query failed for {source_gene}->{target_gene}: status {response.status_code}")
        
        except Exception as e:
            print(f"Error querying {source_gene}->{target_gene}: {str(e)}")
        
        time.sleep(0.5)
    
    if not all_results:
        return pd.DataFrame()
    
    df = pd.DataFrame(all_results)
    
    # Prioritize camelCase column names for display
    # If we have both formats, use camelCase
    rename_map = {}
    for old_name in ['source_genesymbol', 'target_genesymbol', 'is_directed', 
                     'is_stimulation', 'is_inhibition', 'consensus_direction',
                     'consensus_stimulation', 'consensus_inhibition','type']:
        camel_name = old_name.replace('_', ' ').title().replace(' ', '')
        camel_name = camel_name[0].lower() + camel_name[1:]  # Make first letter lowercase
        if old_name in df.columns and camel_name not in df.columns:
            rename_map[old_name] = camel_name
    
    if rename_map:
        df = df.rename(columns=rename_map)
    
    return df

In [555]:
kinase_dict = (
    tape_pred.loc[:, ['kinase_gene', 'kinase']]
    .drop_duplicates()
    .groupby('kinase_gene')['kinase']
    .apply(lambda s: list(s)[0])
    .to_dict())

substrate_dict =  (
    pairs_aug.loc[:, ['substrate_gene', 'substrate']]
    .drop_duplicates()
    .groupby('substrate_gene')['substrate']
    .apply(lambda s: list(s)[0])
    .to_dict())

In [556]:
len(substrate_stats['substrate_gene'].drop_duplicates().tolist())

1130

In [439]:
# Query GSK3B -> CTNNB1 interactions
from tqdm.auto import tqdm
all_df = []
for substrate in tqdm(substrate_stats['substrate_gene'].drop_duplicates().tolist()):
    substrate_stats_test = substrate_stats.loc[substrate_stats['substrate_gene']==substrate,:]
    source = substrate_dict[substrate]
    df_curr_allkinase = query_omnipath_kinase_substrate([(source,source)])
    for i,row in substrate_stats_test.iterrows(): 
        all_kinases = row['high_kinases']
        if all_kinases!='':
            kinase_ls = all_kinases.split('; ')
            for kinase_logit in kinase_ls:
                target = kinase_dict[kinase_logit.split(':')[0]]
                if df_curr_allkinase.shape[0]>0:
                    df_curr_allkinase_1 = df_curr_allkinase.loc[(df_curr_allkinase['source']==source)&(df_curr_allkinase['target']==target),:]
                    df_curr_allkinase_2 = df_curr_allkinase.loc[(df_curr_allkinase['source']==target)&(df_curr_allkinase['target']==source),:]
                    all_df.append(df_curr_allkinase_1)
                    all_df.append(df_curr_allkinase_2)
                    

  0%|          | 0/1133 [00:00<?, ?it/s]

Error querying Q9C0H6->Q9C0H6: HTTPSConnectionPool(host='omnipathdb.org', port=443): Read timed out. (read timeout=15)
Error querying Q9UMN6->Q9UMN6: HTTPSConnectionPool(host='omnipathdb.org', port=443): Read timed out. (read timeout=15)


In [559]:
all_df = pd.concat(all_df)

In [558]:
omnipath_df = all_df.drop_duplicates(['source','target'])

In [444]:
omnipath_df.to_pickle('./260121_omnipath_query.pkl')

In [560]:
# start clean
ss = selective_substrates.copy()

# ensure we don't have duplicate index issues
ss = ss.reset_index(drop=True)

# split to list
ss["high_kinase_list"] = ss["high_kinases"].fillna("").str.split(";")

# explode into long form
ss_long = ss.explode("high_kinase_list", ignore_index=True)

# extract kinase symbol (before first ':')
ss_long["high_kinase"] = (
    ss_long["high_kinase_list"]
    .astype(str)
    .str.strip()
    .replace("", pd.NA)
    .str.split(":")
    .str[0]
)

ss_long = ss_long.dropna(subset=["high_kinase"])

In [561]:
ss_long = (
    selective_substrates
    .reset_index(drop=True)
    .assign(high_kinase=lambda d: d["high_kinases"].fillna("").str.split(";"))
    .explode("high_kinase", ignore_index=True)
)

ss_long["high_kinase"] = (
    ss_long["high_kinase"]
    .astype(str)
    .str.strip()
    .replace("", pd.NA)
    .str.split(":")
    .str[0]
)

ss_long = ss_long.dropna(subset=["high_kinase"])

In [562]:
op = omnipath_df.copy()

# forward edges
op_fwd = op.assign(
    kinase=op["sourceGenesymbol"],
    kinase_acc=op["source"],
    substrate=op["targetGenesymbol"],
    substrate_acc=op["target"],
    direction="source_to_target"
)

# reverse edges
op_rev = op.assign(
    kinase=op["targetGenesymbol"],
    kinase_acc=op["target"],
    substrate=op["sourceGenesymbol"],
    substrate_acc=op["source"],
    direction="target_to_source"
)

op_edges = pd.concat([op_fwd, op_rev], ignore_index=True)



In [563]:
ss_long['high_kinase_acc'] = ss_long['high_kinase'].apply(lambda s: kinase_dict[s])
ss_long['substrate_acc'] = ss_long['substrate_gene'].apply(lambda s: substrate_dict[s])

In [588]:
merged = ss_long.merge(
    op_edges,
    left_on=["high_kinase_acc", "substrate_acc"],
    right_on=["kinase_acc", "substrate_acc"],
    how="inner"
)


In [589]:
print("Original rows:", len(selective_substrates))
print("Exploded rows:", len(ss_long))
print("Matched rows:", len(merged))

merged[["substrate_gene","high_kinase","direction","sourceGenesymbol","targetGenesymbol",'high_kinase_acc']].head()


Original rows: 389
Exploded rows: 4087
Matched rows: 81


,substrate_gene,high_kinase,direction,sourceGenesymbol,targetGenesymbol,high_kinase_acc
0,TP53,PLK2,source_to_target,PLK2,TP53,Q9NYY3
1,TP53,PLK2,target_to_source,TP53,PLK2,Q9NYY3
2,TP53,PLK3,source_to_target,PLK3,TP53,Q9H4B4
3,TP53,PLK3,target_to_source,TP53,PLK3,Q9H4B4
4,TP53,VRK1,source_to_target,VRK1,TP53,Q99986


In [652]:
merged = merged.loc[:,['substrate_seq_15aa', 'substrate_gene','high_kinase', 'Clinical_Significance', 'Disease/Phenotype', 'Review_Status',
       'Accession','p_aa', 'p_ind', 'var_type',
       'var_class', 'dbSNP', 'FTID']]

In [653]:
final_df = merged.copy()

In [654]:
final_df.columns = ['substrate_seq_15aa', 'substrate_gene','kinase_gene', 'Clinical_Significance', 'Disease/Phenotype', 'Review_Status',
       'Accession','p_aa', 'p_ind', 'var_type',
       'var_class', 'dbSNP', 'FTID']

kinase column predicted to phosphorylate the phosphosite in each row defined by p_ind and substrate column, the interactions are filtered to include confidently interacting proteins documented in omnipath, and mutating into a non-phosphorylable amino acid results in the dbSNP column  rsID, which suggests disease consequences in 'Clinical_Significance', 'Disease/Phenotype' columns. for each substrate_seq_15aa, summarize the impact, and try to support the kinase-substrate-disease linkage with literature searches, ignore other columns 

In [611]:
pairs_aug_phosphonetwork = pd.read_csv('../../data/pairs_aug_phosphonetwork.csv').iloc[:,1:]

In [615]:
ss_novel = ss_long.copy()

In [617]:
ss_novel.columns = ['substrate_seq_15aa', 'substrate_gene', 'p_aa', 'p_ind', 'var_type',
       'var_class', 'dbSNP', 'FTID', 'high_kinase_ratio', 'low_kinase_ratio',
       'logit_range', 'high_kinases', 'low_kinases', 'rsID',
       'Clinical_Significance', 'Disease/Phenotype', 'Review_Status',
       'Accession', 'high_kinase', 'kinase', 'substrate']

In [620]:
ss_melt_novel_phosphonetwork = pd.merge(ss_novel,pairs_aug_phosphonetwork.loc[:,['kinase','substrate']].drop_duplicates())

In [625]:
ss_melt_overlap_phosphonetwork_nr = ss_melt_novel_phosphonetwork.drop_duplicates(['p_ind','substrate','kinase'])

In [630]:
ss_melt_overlap_phosphonetwork_nr = ss_melt_overlap_phosphonetwork_nr.loc[:,['high_kinase', 'substrate_seq_15aa', 'substrate_gene', 'p_aa', 'p_ind','kinase', 'substrate', 'var_type',
       'var_class', 'dbSNP', 'FTID', 'high_kinase_ratio', 'low_kinase_ratio',
       'logit_range', 'high_kinases', 'low_kinases', 'rsID',
       'Clinical_Significance', 'Disease/Phenotype', 'Review_Status',
       'Accession']]

In [659]:
ss_melt_overlap_phosphonetwork_nr = ss_melt_overlap_phosphonetwork_nr.loc[:,['substrate_seq_15aa', 'substrate_gene','high_kinase', 'Clinical_Significance', 'Disease/Phenotype', 'Review_Status',
       'Accession','p_aa', 'p_ind', 'var_type',
       'var_class', 'dbSNP', 'FTID']]
ss_melt_overlap_phosphonetwork_nr.columns = ['substrate_seq_15aa', 'substrate_gene','kinase_gene', 'Clinical_Significance', 'Disease/Phenotype', 'Review_Status',
       'Accession','p_aa', 'p_ind', 'var_type',
       'var_class', 'dbSNP', 'FTID']

In [631]:
import requests
import time
import sys
from typing import List, Tuple, Dict, Optional

class StringDBQuery:
    """Class to handle STRING-DB API queries for kinase-substrate pairs"""
    
    def __init__(self, species: int = 9606, version: str = "12.0"):
        """
        Initialize STRING-DB query handler
        
        Args:
            species: NCBI taxon ID (default: 9606 for human)
            version: STRING-DB version to use (default: "12.0")
        """
        self.species = species
        self.base_url = f"https://version-{version.replace('.', '-')}.string-db.org/api"
        self.caller_identity = "kinase_substrate_query"
        
    def map_uniprot_to_string(self, uniprot_ids: List[str]) -> Dict[str, str]:
        """
        Map UniProt accession IDs to STRING identifiers
        
        Args:
            uniprot_ids: List of UniProt accession IDs
            
        Returns:
            Dictionary mapping UniProt IDs to STRING IDs
        """
        print(f"Mapping {len(uniprot_ids)} UniProt IDs to STRING IDs...")
        
        request_url = f"{self.base_url}/tsv/get_string_ids"
        
        params = {
            "identifiers": "\r".join(uniprot_ids),
            "species": self.species,
            "limit": 1,  # Get only the best match
            "echo_query": 1,
            "caller_identity": self.caller_identity
        }
        
        response = requests.post(request_url, data=params)
        
        if response.status_code != 200:
            print(f"Error mapping IDs: HTTP {response.status_code}")
            return {}
        
        mapping = {}
        for line in response.text.strip().split("\n")[1:]:  # Skip header
            parts = line.split("\t")
            if len(parts) >= 3:
                query_id = parts[0]
                string_id = parts[2]
                mapping[query_id] = string_id
        
        print(f"Successfully mapped {len(mapping)}/{len(uniprot_ids)} IDs")
        return mapping
    
    def get_interaction(self, protein_a: str, protein_b: str) -> Optional[Dict]:
        """
        Get interaction information between two proteins
        
        Args:
            protein_a: STRING ID for first protein
            protein_b: STRING ID for second protein
            
        Returns:
            Dictionary with interaction data or None if no interaction exists
        """
        request_url = f"{self.base_url}/json/network"
        
        params = {
            "identifiers": f"{protein_a}\r{protein_b}",
            "species": self.species,
            "required_score": 0,  # Get all interactions regardless of score
            "caller_identity": self.caller_identity
        }
        
        response = requests.post(request_url, data=params)
        
        if response.status_code != 200:
            return None
        
        data = response.json()
        
        # Find the interaction between our two proteins
        for interaction in data:
            if (interaction['stringId_A'] == protein_a and interaction['stringId_B'] == protein_b) or \
               (interaction['stringId_A'] == protein_b and interaction['stringId_B'] == protein_a):
                return interaction
        
        return None
    
    def query_kinase_substrate_pairs(
        self, 
        pairs: List[Tuple[str, str]], 
        delay: float = 1.0
    ) -> List[Dict]:
        """
        Query STRING-DB for kinase-substrate pairs
        
        Args:
            pairs: List of (kinase_uniprot_id, substrate_uniprot_id) tuples
            delay: Delay between API calls in seconds (to avoid overloading server)
            
        Returns:
            List of dictionaries containing interaction results
        """
        # Collect all unique UniProt IDs
        all_uniprot_ids = list(set([p[0] for p in pairs] + [p[1] for p in pairs]))
        
        # Map UniProt IDs to STRING IDs
        uniprot_to_string = self.map_uniprot_to_string(all_uniprot_ids)
        
        results = []
        
        print(f"\nQuerying {len(pairs)} kinase-substrate pairs...")
        
        for i, (kinase_uniprot, substrate_uniprot) in enumerate(pairs, 1):
            # Get STRING IDs
            kinase_string = uniprot_to_string.get(kinase_uniprot)
            substrate_string = uniprot_to_string.get(substrate_uniprot)
            
            if not kinase_string or not substrate_string:
                result = {
                    'kinase_uniprot': kinase_uniprot,
                    'substrate_uniprot': substrate_uniprot,
                    'kinase_string': kinase_string,
                    'substrate_string': substrate_string,
                    'interaction_found': False,
                    'error': 'Failed to map one or both UniProt IDs to STRING'
                }
                results.append(result)
                continue
            
            # Query interaction
            interaction = self.get_interaction(kinase_string, substrate_string)
            
            if interaction:
                result = {
                    'kinase_uniprot': kinase_uniprot,
                    'substrate_uniprot': substrate_uniprot,
                    'kinase_string': kinase_string,
                    'kinase_name': interaction['preferredName_A'] if interaction['stringId_A'] == kinase_string else interaction['preferredName_B'],
                    'substrate_string': substrate_string,
                    'substrate_name': interaction['preferredName_B'] if interaction['stringId_A'] == kinase_string else interaction['preferredName_A'],
                    'interaction_found': True,
                    'combined_score': interaction['score'],
                    'neighborhood_score': interaction['nscore'],
                    'fusion_score': interaction['fscore'],
                    'cooccurrence_score': interaction['pscore'],
                    'coexpression_score': interaction['ascore'],
                    'experimental_score': interaction['escore'],
                    'database_score': interaction['dscore'],
                    'textmining_score': interaction['tscore']
                }
            else:
                result = {
                    'kinase_uniprot': kinase_uniprot,
                    'substrate_uniprot': substrate_uniprot,
                    'kinase_string': kinase_string,
                    'substrate_string': substrate_string,
                    'interaction_found': False,
                    'error': None
                }
            
            results.append(result)
            
            # Progress update
            if i % 10 == 0:
                print(f"Processed {i}/{len(pairs)} pairs...")
            
            # Delay to be polite to the server
            if i < len(pairs):
                time.sleep(delay)
        
        print(f"Completed querying {len(pairs)} pairs")
        return results


def print_results(results: List[Dict]):
    """Print results in a readable format"""
    
    print("\n" + "="*80)
    print("RESULTS SUMMARY")
    print("="*80)
    
    total = len(results)
    found = sum(1 for r in results if r['interaction_found'])
    
    print(f"\nTotal pairs queried: {total}")
    print(f"Interactions found: {found} ({100*found/total:.1f}%)")
    print(f"No interaction: {total - found} ({100*(total-found)/total:.1f}%)")
    
    print("\n" + "="*80)
    print("DETAILED RESULTS")
    print("="*80)
    
    for i, result in enumerate(results, 1):
        print(f"\n{i}. {result['kinase_uniprot']} → {result['substrate_uniprot']}")
        
        if result['interaction_found']:
            print(f"   ✓ Interaction found!")
            print(f"   Kinase: {result.get('kinase_name', 'N/A')} ({result['kinase_string']})")
            print(f"   Substrate: {result.get('substrate_name', 'N/A')} ({result['substrate_string']})")
            print(f"   Combined Score: {result['combined_score']}")
            print(f"   Experimental Score: {result['experimental_score']}")
            print(f"   Database Score: {result['database_score']}")
            print(f"   Textmining Score: {result['textmining_score']}")
        else:
            error_msg = result.get('error', 'No interaction found in STRING-DB')
            print(f"   ✗ {error_msg}")


def save_results_to_tsv(results: List[Dict], filename: str = "stringdb_results.tsv"):
    """Save results to a TSV file"""
    
    with open(filename, 'w') as f:
        # Write header
        headers = [
            'kinase_uniprot', 'substrate_uniprot',
            'kinase_string', 'substrate_string',
            'kinase_name', 'substrate_name',
            'interaction_found', 'combined_score',
            'experimental_score', 'database_score',
            'textmining_score', 'coexpression_score',
            'neighborhood_score', 'fusion_score',
            'cooccurrence_score', 'error'
        ]
        f.write('\t'.join(headers) + '\n')
        
        # Write data
        for result in results:
            row = [
                result.get('kinase_uniprot', ''),
                result.get('substrate_uniprot', ''),
                result.get('kinase_string', ''),
                result.get('substrate_string', ''),
                result.get('kinase_name', ''),
                result.get('substrate_name', ''),
                str(result.get('interaction_found', False)),
                str(result.get('combined_score', '')),
                str(result.get('experimental_score', '')),
                str(result.get('database_score', '')),
                str(result.get('textmining_score', '')),
                str(result.get('coexpression_score', '')),
                str(result.get('neighborhood_score', '')),
                str(result.get('fusion_score', '')),
                str(result.get('cooccurrence_score', '')),
                result.get('error', '')
            ]
            f.write('\t'.join(row) + '\n')
    
    print(f"\nResults saved to {filename}")



    

In [632]:
ss_novel_ks = ss_novel.drop_duplicates(['kinase','substrate'])

In [633]:
example_pairs = list(zip(ss_novel['kinase'],ss_novel['substrate']))
    
print("STRING-DB Kinase-Substrate Query Tool")
print("="*80)
print(f"Species: Human (NCBI Taxon ID: 9606)")
print(f"Example pairs to query: {len(example_pairs)}")

# Initialize query handler
query = StringDBQuery(species=9606)

# Query the pairs
results = query.query_kinase_substrate_pairs(example_pairs, delay=1.0)

STRING-DB Kinase-Substrate Query Tool
Species: Human (NCBI Taxon ID: 9606)
Example pairs to query: 4087
Mapping 737 UniProt IDs to STRING IDs...
Successfully mapped 735/737 IDs

Querying 4087 kinase-substrate pairs...
Processed 10/4087 pairs...
Processed 20/4087 pairs...
Processed 30/4087 pairs...
Processed 40/4087 pairs...
Processed 50/4087 pairs...
Processed 60/4087 pairs...
Processed 70/4087 pairs...
Processed 80/4087 pairs...
Processed 90/4087 pairs...
Processed 100/4087 pairs...
Processed 110/4087 pairs...
Processed 120/4087 pairs...
Processed 130/4087 pairs...
Processed 140/4087 pairs...
Processed 150/4087 pairs...
Processed 160/4087 pairs...
Processed 170/4087 pairs...
Processed 180/4087 pairs...
Processed 190/4087 pairs...
Processed 200/4087 pairs...
Processed 210/4087 pairs...
Processed 220/4087 pairs...
Processed 230/4087 pairs...
Processed 240/4087 pairs...
Processed 260/4087 pairs...
Processed 270/4087 pairs...
Processed 290/4087 pairs...
Processed 300/4087 pairs...
Process

In [634]:
stringdb_df = pd.DataFrame(results)

In [635]:
stringdb_df_08 = stringdb_df.loc[(stringdb_df['combined_score']>0.8),:]

In [637]:
ss_melt_novel_stringdb = pd.merge(ss_novel,stringdb_df_08,left_on = ['kinase','substrate'],
        right_on = ['kinase_uniprot','substrate_uniprot'])

In [641]:
ss_melt_overlap_stringdb_nr = ss_melt_novel_stringdb.drop_duplicates(['p_ind','substrate','kinase'])

In [658]:
ss_melt_overlap_stringdb_nr = ss_melt_overlap_stringdb_nr.loc[:,['substrate_seq_15aa', 'substrate_gene','high_kinase', 'Clinical_Significance', 'Disease/Phenotype', 'Review_Status',
       'Accession','p_aa', 'p_ind', 'var_type',
       'var_class', 'dbSNP', 'FTID']]
ss_melt_overlap_stringdb_nr.columns = ['substrate_seq_15aa', 'substrate_gene','kinase_gene', 'Clinical_Significance', 'Disease/Phenotype', 'Review_Status',
       'Accession','p_aa', 'p_ind', 'var_type',
       'var_class', 'dbSNP', 'FTID']

In [663]:
ss_melt_overlap_phosphonetwork_nr['db_source'] = 'phosphonetwork'
ss_melt_overlap_stringdb_nr['db_source'] = 'stringdb'
final_df['db_source'] = 'omnipath'

In [669]:
ss_melt_overlap_alldb = pd.concat([ss_melt_overlap_phosphonetwork_nr,
          ss_melt_overlap_stringdb_nr,
          final_df])

In [670]:
ss_melt_overlap_alldb = ss_melt_overlap_alldb.loc[:,['kinase_gene','substrate_gene','p_ind','p_aa','dbSNP','Clinical_Significance',
                                                                    'Disease/Phenotype','db_source']].drop_duplicates(['kinase_gene','substrate_gene','p_ind'])
ss_melt_overlap_alldb = ss_melt_overlap_alldb.loc[~ss_melt_overlap_alldb['Clinical_Significance'].isna(),:]
ss_melt_overlap_alldb = ss_melt_overlap_alldb.loc[ss_melt_overlap_alldb['Clinical_Significance'].apply(lambda s: 'pathogenic' in s.lower()),:]
ss_melt_overlap_alldb = ss_melt_overlap_alldb.loc[~(ss_melt_overlap_alldb['kinase_gene'] == 'CaMK1a'),:]

In [672]:
ss_melt_overlap_alldb.groupby('db_source').count()

,kinase_gene,substrate_gene,p_ind,p_aa,dbSNP,Clinical_Significance,Disease/Phenotype
db_source,,,,,,,
omnipath,31,31,31,31,31,31,31
phosphonetwork,4,4,4,4,4,4,4
stringdb,31,31,31,31,31,31,31


In [674]:
ss_melt_overlap_alldb.to_csv('260128_ptmvar_psites_overlap_dbs.csv')